In [1]:
# If not installed
# !pip install pyarrow pandas

import pandas as pd
import pyarrow.dataset as ds

bau_path = "prod_output.parquet"
test_path = "test_output.parquet"

bau_ds = ds.dataset(bau_path, format="parquet")
test_ds = ds.dataset(test_path, format="parquet")

In [2]:
def process_batches(dataset, is_test=False):

    results = []

    for batch in dataset.to_batches(batch_size=500_000):
        df = batch.to_pandas()

        # ✅ Keep required columns
        df = df[[
            "UPC_CODE",
            "PRODUCT_DESCRIPTION",
            "UNITS_SOLD",
            "TOTAL_DOLLARS"
        ]]

        # ✅ Convert numeric (safety)
        df["UNITS_SOLD"] = pd.to_numeric(df["UNITS_SOLD"], errors="coerce").fillna(0)
        df["TOTAL_DOLLARS"] = pd.to_numeric(df["TOTAL_DOLLARS"], errors="coerce").fillna(0)

        # ✅ Group within chunk
        agg = df.groupby(
            ["UPC_CODE", "PRODUCT_DESCRIPTION"],
            as_index=False
        ).agg(
            UNITS=("UNITS_SOLD", "sum"),
            DOLLARS=("TOTAL_DOLLARS", "sum")
        )

        results.append(agg)

    # ✅ Combine all chunks and re-aggregate
    final_df = pd.concat(results, ignore_index=True)

    final_df = final_df.groupby(
        ["UPC_CODE", "PRODUCT_DESCRIPTION"],
        as_index=False
    ).sum()

    return final_df

In [3]:
bau_summary = process_batches(bau_ds)
bau_summary.head()


,UPC_CODE,PRODUCT_DESCRIPTION,UNITS,DOLLARS
0,10012587788346,GLAD CLING WRAP 45 SQFT,29738.0,44603.25
1,1001822,FD PARTY MISC $1,1.0,1.00
2,10024600011980,MORTON SEA SALT 4.4Z,11850.0,17773.75
3,10024600017548,MORTON SEASON ALL 3.25Z,8816.0,13222.25
4,10029839700186,FRNLY CELEB CAKE SINGLE 8.5Z,6366.0,12731.50


In [4]:
test_summary = process_batches(test_ds)
test_summary.head()

,UPC_CODE,PRODUCT_DESCRIPTION,UNITS,DOLLARS
0,,MISC,1081.0,10.81
1,,MISC $1.00,1456.0,1456.00
2,1,DSTY NFL BEANIE BEARS PRO DUNN,-4.0,-1.28
3,10,PAPER BAG CHARGE,4337.0,435.10
4,10000,MD COLOR ME ART PRNCSS/PIRATE,3.0,0.75


In [5]:
bau_summary = bau_summary.rename(columns={
    "UNITS": "BAU Units",
    "DOLLARS": "BAU Dollars"
})

test_summary = test_summary.rename(columns={
    "UNITS": "TEST Units",
    "DOLLARS": "TEST Dollars"
})

In [6]:
comparison_df = pd.merge(
    bau_summary,
    test_summary,
    on=["UPC_CODE", "PRODUCT_DESCRIPTION"],
    how="inner"   # ✅ match manual logic
)

In [7]:
comparison_df["Units Difference"] = (
    comparison_df["TEST Units"] - comparison_df["BAU Units"]
)

comparison_df["Dollar Difference"] = (
    comparison_df["TEST Dollars"] - comparison_df["BAU Dollars"]
)

comparison_df["Unit % Difference"] = (
    comparison_df["Units Difference"] / comparison_df["BAU Units"]
).replace([float("inf"), -float("inf")], 0).fillna(0)

comparison_df["Dollar % Difference"] = (
    comparison_df["Dollar Difference"] / comparison_df["BAU Dollars"]
).replace([float("inf"), -float("inf")], 0).fillna(0)

In [9]:
comparison_df = comparison_df.drop_duplicates()
comparison_df.shape[0]


37268

In [10]:
comparison_df = pd.merge(
    bau_summary,
    test_summary,
    on=["UPC_CODE", "PRODUCT_DESCRIPTION"],
    how="outer"
)

In [11]:
comparison_df["Present In"] = "Present in Both"

comparison_df.loc[
    comparison_df["BAU Units"].isna(),
    "Present In"
] = "Present only in TEST"

comparison_df.loc[
    comparison_df["TEST Units"].isna(),
    "Present In"
] = "Present only in BAU"

In [ ]:

comparison_df = comparison_df.drop_duplicates()
comparison_df.dropna(subset=['UPC_CODE'])
comparison_df.shape[0]
# comparison_df.columns

Index(['UPC_CODE', 'PRODUCT_DESCRIPTION', 'BAU Units', 'BAU Dollars',
       'TEST Units', 'TEST Dollars', 'Present In'],
      dtype='str')

In [23]:
both_df_1 = comparison_df[
    comparison_df["Present In"] == "Present only in BAU"
]
both_df_2 = comparison_df[
    comparison_df["Present In"] == "Present only in TEST"
]
both_df_3 = comparison_df[
    comparison_df["Present In"] == "Present in Both"
]


# both_df_1.shape[0]
# both_df_2.shape[0]
both_df_3.shape[0]

37268

In [24]:
bau_summary["KEY_UPC"] = bau_summary["UPC_CODE"].astype(str)
test_summary["KEY_UPC"] = test_summary["UPC_CODE"].astype(str)

common_upc = set(bau_summary["KEY_UPC"]).intersection(set(test_summary["KEY_UPC"]))

# now check description mismatch inside those
merged_upc = pd.merge(
    bau_summary,
    test_summary,
    on="UPC_CODE",
    how="inner",
    suffixes=("_BAU", "_TEST")
)

mismatch = merged_upc[
    merged_upc["PRODUCT_DESCRIPTION_BAU"] != merged_upc["PRODUCT_DESCRIPTION_TEST"]
]

print("UPC matches but description mismatch:", len(mismatch))

UPC matches but description mismatch: 14140


In [25]:
# Exact keys
bau_keys = set(
    zip(bau_summary["UPC_CODE"], bau_summary["PRODUCT_DESCRIPTION"])
)

test_keys = set(
    zip(test_summary["UPC_CODE"], test_summary["PRODUCT_DESCRIPTION"])
)

# Strict matches
common_keys = bau_keys & test_keys

# Only BAU
bau_only = bau_keys - test_keys

# Only TEST
test_only = test_keys - bau_keys

print("BAU Only:", len(bau_only))
print("TEST Only:", len(test_only))
print("Both:", len(common_keys))

BAU Only: 25946
TEST Only: 84974
Both: 37268


In [26]:
# Get sets
bau_upc = set(bau_summary["UPC_CODE"])
test_upc = set(test_summary["UPC_CODE"])

common_upc = bau_upc & test_upc

problem_upc = []

for upc in common_upc:
    bau_desc = set(
        bau_summary.loc[
            bau_summary["UPC_CODE"] == upc, "PRODUCT_DESCRIPTION"
        ]
    )
    test_desc = set(
        test_summary.loc[
            test_summary["UPC_CODE"] == upc, "PRODUCT_DESCRIPTION"
        ]
    )

    # ✅ No overlap between descriptions
    if bau_desc.isdisjoint(test_desc):
        problem_upc.append(upc)

print("Total problem UPCs:", len(problem_upc))

Total problem UPCs: 13896


In [28]:
report_rows = []

for upc in problem_upc:

    bau_slice = bau_summary[bau_summary["UPC_CODE"] == upc]
    test_slice = test_summary[test_summary["UPC_CODE"] == upc]

    # Collect descriptions
    bau_desc_list = sorted(bau_slice["PRODUCT_DESCRIPTION"].unique())
    test_desc_list = sorted(test_slice["PRODUCT_DESCRIPTION"].unique())

    # Aggregate values
    bau_units = bau_slice["BAU Units"].sum()
    test_units = test_slice["TEST Units"].sum()

    bau_dollars = bau_slice["BAU Dollars"].sum()
    test_dollars = test_slice["TEST Dollars"].sum()

    report_rows.append({
        "UPC_CODE": upc,
        "BAU_DESCRIPTIONS": " | ".join(bau_desc_list),
        "TEST_DESCRIPTIONS": " | ".join(test_desc_list),
        "BAU_TOTAL_UNITS": bau_units,
        "TEST_TOTAL_UNITS": test_units,
        "BAU_TOTAL_DOLLARS": bau_dollars,
        "TEST_TOTAL_DOLLARS": test_dollars
    })

mismatch_report = pd.DataFrame(report_rows)
mismatch_report.shape[0]

13896

In [16]:
import pandas as pd
import pyarrow.dataset as ds

In [17]:
data_path = r'C:\Users\armu6001\Downloads\cyril_app\retailer_output.parquet'
retailer_ds = ds.dataset(data_path, format="parquet")

In [18]:
def process_batches_2(dataset, is_test=False):

    results = []

    for batch in dataset.to_batches(batch_size=500_000):
        df = batch.to_pandas()

        results.append(df)
    # ✅ Combine all chunks and re-aggregate
    final_df = pd.concat(results, ignore_index=True)

    return final_df

In [19]:
retailer_df = process_batches_2(retailer_ds)
retailer_df_filtered = retailer_df[
    retailer_df["Store"] == "00238"
    ] 
retailer_df_filtered.head()

,Flag,Store,UPC,Description,SizeDescription,Case,PriceMultiplier,Price,UnitsSold
1633458,U,00238,00002000044998,GG STMRS BRUSSELS SPROUTS S&P,9 OZ,8,1,1.99,15
1633459,U,00238,00002310011024,SHEBA PERF PORT SINGL WTFH&TUN,2.6 OZ,24,1,.72,8
1633460,U,00238,00087897100001,MOCO DE GORILA ROCKERO SQUIZZ,11.9OZ,1,1,2.68,1
1633461,U,00238,00002430004102,LITTLE DEB HONEY BUNS,10.6OZ,1,1,2.51,171
1633462,U,00238,00004138309073,LACTAID 100 WHOLE MILK,96 OZ,6,1,5.48,18


In [21]:
def to_numeric_store(series):
    return pd.to_numeric(series, errors="coerce") 

retailer_df_filtered["Store"] = pd.to_numeric(retailer_df_filtered["Store"].squeeze(), errors="coerce").fillna(0)

retailer_df_filtered.head()

,Flag,Store,UPC,Description,SizeDescription,Case,PriceMultiplier,Price,UnitsSold
1633458,U,238,00002000044998,GG STMRS BRUSSELS SPROUTS S&P,9 OZ,8,1,1.99,15
1633459,U,238,00002310011024,SHEBA PERF PORT SINGL WTFH&TUN,2.6 OZ,24,1,.72,8
1633460,U,238,00087897100001,MOCO DE GORILA ROCKERO SQUIZZ,11.9OZ,1,1,2.68,1
1633461,U,238,00002430004102,LITTLE DEB HONEY BUNS,10.6OZ,1,1,2.51,171
1633462,U,238,00004138309073,LACTAID 100 WHOLE MILK,96 OZ,6,1,5.48,18
